# OpenAI + ChromaDB

In [ ]:
# pip install -U langchain langchain-openai langchain-chroma chromadb

import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor


# --------------------------------------------------
# 1. API KEY
# --------------------------------------------------

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"


# --------------------------------------------------
# 2. LLM
# --------------------------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


# --------------------------------------------------
# 3. Embedding Model
# --------------------------------------------------

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# --------------------------------------------------
# 4. Documents
# --------------------------------------------------

documents = [

    Document(
        page_content="""
        Transformers are neural network architectures widely used
        in Natural Language Processing. They use attention mechanisms
        to understand relationships between tokens.
        """
    ),

    Document(
        page_content="""
        Self-attention allows every token in a sequence to interact
        with other tokens. It uses Query, Key and Value vectors to
        calculate attention scores.
        """
    ),

    Document(
        page_content="""
        Positional encoding provides information about the position
        of tokens because Transformers do not process tokens
        sequentially like traditional RNNs.
        """
    ),

    Document(
        page_content="""
        RNNs process sequences sequentially and maintain a hidden
        state that carries information from previous time steps.
        """
    ),

]


# --------------------------------------------------
# 5. Create Chroma Vector Store
# --------------------------------------------------

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="transformer_documents"
)


# --------------------------------------------------
# 6. Base Retriever
# --------------------------------------------------

base_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# --------------------------------------------------
# 7. Create Compressor
# --------------------------------------------------

compressor = LLMChainExtractor.from_llm(
    llm
)


# --------------------------------------------------
# 8. Contextual Compression Retriever
# --------------------------------------------------

compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)


# --------------------------------------------------
# 9. Query
# --------------------------------------------------

query = "What is self-attention in Transformers?"


# --------------------------------------------------
# 10. Retrieve Compressed Documents
# --------------------------------------------------

docs = compression_retriever.invoke(query)


# --------------------------------------------------
# 11. Display Results
# --------------------------------------------------

print("\nCompressed Documents:\n")

for i, doc in enumerate(docs, start=1):

    print(f"--- Document {i} ---")

    print(doc.page_content)

    print()